# Integrated LSC Regression Tables

This notebook collects the trend-model outputs from the four completed scalar LSC analyses and turns them into report-ready regression tables. It does not refit models. The individual measure notebooks remain the source of annual estimates and trend regressions; this notebook only harmonises and formats their outputs.

The main-text table foregrounds ADHD and Autism target trajectories by frame. The appendix table reports the unframed baseline comparator trajectories in the same compact style.


## Setup

All table outputs are written under `reports/tables/lsc/regression/`. The LaTeX exports use plain `booktabs` syntax so they can be included in the dissertation without adding a table-specific package.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "reports").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "reports").exists())

OUTPUT_DIR = PROJECT_ROOT / "reports" / "tables" / "lsc" / "regression"
DIAGNOSTIC_OUTPUT_DIR = PROJECT_ROOT / "reports" / "tables" / "lsc" / "diagnostics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = OUTPUT_DIR / "lsc_regression_models_combined.csv"
TARGET_FRAME_CSV_PATH = OUTPUT_DIR / "lsc_regression_target_frames.csv"
TARGET_FRAME_TEX_PATH = OUTPUT_DIR / "lsc_regression_target_frames.tex"
BASELINE_CSV_PATH = OUTPUT_DIR / "lsc_regression_baseline_comparators.csv"
BASELINE_TEX_PATH = OUTPUT_DIR / "lsc_regression_baseline_comparators.tex"
CLASSIFICATION_TREND_CSV_PATH = PROJECT_ROOT / "reports" / "tables" / "lsc" / "classification" / "lsc_classification_frame_time_trends.csv"
AR1_SENSITIVITY_CSV_PATH = DIAGNOSTIC_OUTPUT_DIR / "lsc_ar1_sensitivity_flagged.csv"
AR1_SENSITIVITY_TEX_PATH = DIAGNOSTIC_OUTPUT_DIR / "lsc_ar1_sensitivity_flagged.tex"

MEASURE_CONFIGS = [
    {
        "measure": "Salience",
        "order": 1,
        "year_basis": "source year",
        "path": PROJECT_ROOT / "data" / "processed" / "lsc" / "salience" / "salience_trend_models.csv",
        "overall_stratum": "source_year_salience",
    },
    {
        "measure": "Sentiment",
        "order": 2,
        "year_basis": "publication year",
        "path": PROJECT_ROOT / "data" / "processed" / "lsc" / "sentiment" / "lsc_sentiment_trend_models.csv",
        "overall_stratum": "substantive_core_overall",
    },
    {
        "measure": "Intensity",
        "order": 3,
        "year_basis": "publication year",
        "path": PROJECT_ROOT / "data" / "processed" / "lsc" / "intensity" / "lsc_intensity_trend_models.csv",
        "overall_stratum": "substantive_core_overall",
    },
    {
        "measure": "Breadth",
        "order": 4,
        "year_basis": "publication year",
        "path": PROJECT_ROOT / "data" / "processed" / "lsc" / "breadth" / "lsc_breadth_trend_models.csv",
        "overall_stratum": "substantive_core_overall",
    },
]

TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
TARGET_FRAME_STRATA = ["substantive_core_overall", "clinical_only", "lived_only"]
SALIENCE_TARGET_STRATA = ["source_year_salience"]

REQUIRED_COLUMNS = {
    "analysis_unit",
    "frame_stratum",
    "index_name",
    "n_years",
    "linear_slope_per_year",
    "linear_slope_se",
    "linear_p_value",
    "linear_adj_r_squared",
    "standardized_beta_year",
    "durbin_watson",
    "lag1_residual_autocorrelation",
    "autocorrelation_flag",
    "ar1_sensitivity_slope_per_year",
    "ar1_sensitivity_p_value",
    "quadratic_delta_adj_r_squared",
}


## Load Trend Models

The four measure notebooks already write the necessary regression information. Salience uses `analysis_role`; the semantic notebooks use `term_role`. This cell harmonises that naming and adds measure-level metadata.

In [2]:
frames = []
missing_inputs = [config["path"] for config in MEASURE_CONFIGS if not config["path"].exists()]
if missing_inputs:
    missing = "\n".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_inputs)
    raise FileNotFoundError(f"Missing trend-model input(s):\n{missing}")

for config in MEASURE_CONFIGS:
    trend_models = pd.read_csv(config["path"])
    missing_columns = sorted(REQUIRED_COLUMNS - set(trend_models.columns))
    if missing_columns:
        raise ValueError(f"{config['path'].relative_to(PROJECT_ROOT)} is missing columns: {missing_columns}")

    if "term_role" not in trend_models.columns and "analysis_role" in trend_models.columns:
        trend_models = trend_models.rename(columns={"analysis_role": "term_role"})
    if "target_group" not in trend_models.columns:
        trend_models["target_group"] = np.where(
            trend_models["term_role"].eq("target"),
            trend_models["analysis_unit"],
            "baseline",
        )

    trend_models["measure"] = config["measure"]
    trend_models["measure_order"] = config["order"]
    trend_models["year_basis"] = config["year_basis"]
    trend_models["source_path"] = str(config["path"].relative_to(PROJECT_ROOT))
    frames.append(trend_models)

combined = pd.concat(frames, ignore_index=True)
combined["autocorrelation_flag"] = combined["autocorrelation_flag"].astype(bool)
combined = combined.sort_values(["measure_order", "term_role", "analysis_unit", "frame_stratum"]).reset_index(drop=True)

combined_columns = [
    "measure", "measure_order", "analysis_unit", "term_role", "target_group", "frame_stratum",
    "year_basis", "index_name", "n_years", "year_center", "linear_intercept",
    "linear_slope_per_year", "linear_slope_se", "linear_p_value", "linear_r_squared",
    "linear_adj_r_squared", "standardized_beta_year", "durbin_watson",
    "lag1_residual_autocorrelation", "autocorrelation_flag", "ar1_sensitivity_slope_per_year",
    "ar1_sensitivity_p_value", "quadratic_adj_r_squared", "quadratic_delta_adj_r_squared",
    "source_path",
]
combined[combined_columns].to_csv(COMBINED_PATH, index=False)

print(f"Wrote {COMBINED_PATH.relative_to(PROJECT_ROOT)}")
display(combined[["measure", "analysis_unit", "term_role", "frame_stratum", "linear_slope_per_year", "linear_p_value", "autocorrelation_flag"]])


Wrote reports/tables/lsc/regression/lsc_regression_models_combined.csv


,measure,analysis_unit,term_role,frame_stratum,linear_slope_per_year,linear_p_value,autocorrelation_flag
0,Salience,frustration,baseline,source_year_salience,-0.017517,0.317958,True
1,Salience,loneliness,baseline,source_year_salience,0.005580,0.022763,False
2,Salience,sadness,baseline,source_year_salience,-0.017072,0.001056,False
3,Salience,ADHD,target,source_year_salience,0.007272,0.117824,False
4,Salience,Autism,target,source_year_salience,-0.022807,0.024012,False
5,Sentiment,frustration,baseline,unframed_baseline,0.002287,0.014441,True
6,Sentiment,loneliness,baseline,unframed_baseline,0.002453,0.012791,True
7,Sentiment,sadness,baseline,unframed_baseline,-0.001604,0.009570,False
8,Sentiment,ADHD,target,clinical_only,0.000427,0.631173,True
9,Sentiment,ADHD,target,lived_only,0.000342,0.712171,False


## Formatting Helpers

The table cells show annual unstandardised OLS slopes as `B(SE)`, with significance stars and a dagger for residual-autocorrelation flags. Standardised beta appears in the report-facing tables; AR(1) sensitivity estimates for flagged rows are reported in a dedicated appendix table.


In [3]:
def star_marker(p_value: float) -> str:
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


def format_number(value: float, digits: int) -> str:
    if pd.isna(value):
        return ""
    value = float(value)
    if abs(value) < 0.5 * 10 ** (-digits):
        value = 0.0
    return f"{value:.{digits}f}"


def format_p_value(value: float) -> str:
    if pd.isna(value):
        return ""
    value = float(value)
    if value < 0.001:
        return r"$<.001$"
    return "$" + f"{value:.3f}".replace("0.", ".") + "$"


def coefficient_plain(row: pd.Series) -> str:
    markers = star_marker(row["linear_p_value"])
    if bool(row["autocorrelation_flag"]):
        markers += "^dagger"
    return f"{format_number(row['linear_slope_per_year'], 4)}{markers} ({format_number(row['linear_slope_se'], 4)})"


def coefficient_latex(row: pd.Series) -> str:
    markers = star_marker(row["linear_p_value"])
    if bool(row["autocorrelation_flag"]):
        markers += r"\dagger"
    slope = format_number(row["linear_slope_per_year"], 4)
    se = format_number(row["linear_slope_se"], 4)
    if markers:
        return f"${slope}^{{{markers}}}$ ({se})"
    return f"${slope}$ ({se})"


def display_stratum(frame_stratum: str) -> str:
    labels = {
        "source_year_salience": "Overall",
        "substantive_core_overall": "Overall",
        "clinical_only": "Clinical",
        "lived_only": "Lived experience",
        "unframed_baseline": "Comparator",
    }
    return labels.get(frame_stratum, frame_stratum.replace("_", " ").title())


def latex_escape(value: object) -> str:
    text = str(value)
    return (
        text.replace("\\", r"\textbackslash{}")
        .replace("&", r"\&")
        .replace("%", r"\%")
        .replace("$", r"\$")
        .replace("#", r"\#")
        .replace("_", r"\_")
    )


def build_latex_table(
    *,
    caption: str,
    label: str,
    headers: list[str],
    rows: list[list[str]],
    alignment: str,
    size: str,
    note: str,
) -> str:
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        f"\\{size}",
        f"\\begin{{tabular}}{{@{{}}{alignment}@{{}}}}",
        r"\toprule",
        " & ".join(headers) + r" \\",
        r"\midrule",
    ]
    for row in rows:
        lines.append(" & ".join(row) + r" \\")
    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\parbox{\linewidth}{\footnotesize " + note + "}",
        r"\end{table}",
        "",
    ])
    return "\n".join(lines)


## Main Text Target-Frame Table

The main table reports ADHD and Autism target trends by frame. Salience has only an Overall source-year row because the salience denominator is not frame-stratified; the three semantic measures report Overall, Clinical, and Lived-experience trajectories.


In [4]:
target_parts = []
for config in MEASURE_CONFIGS:
    measure_rows = combined.loc[combined["measure"].eq(config["measure"])]
    strata = SALIENCE_TARGET_STRATA if config["measure"] == "Salience" else TARGET_FRAME_STRATA
    target_mask = (
        measure_rows["term_role"].eq("target")
        & measure_rows["analysis_unit"].isin(TARGET_UNITS)
        & measure_rows["frame_stratum"].isin(strata)
    )
    target_parts.append(measure_rows.loc[target_mask].copy())

target_table = pd.concat(target_parts, ignore_index=True)
target_table["unit_order"] = target_table["analysis_unit"].map({"ADHD": 1, "Autism": 2}).fillna(99)
target_table["frame_order"] = target_table["frame_stratum"].map({
    "source_year_salience": 1,
    "substantive_core_overall": 1,
    "clinical_only": 2,
    "lived_only": 3,
}).fillna(99)
target_table = target_table.sort_values(["measure_order", "unit_order", "frame_order"]).reset_index(drop=True)

if len(target_table) != 20:
    raise AssertionError(f"Target-frame regression table should have 20 rows, found {len(target_table)}")
if target_table["term_role"].ne("target").any():
    raise AssertionError("Main target-frame table should contain target rows only")
if target_table["frame_stratum"].eq("mixed").any():
    raise AssertionError("Mixed-frame rows should not appear in the main target-frame table")
if combined.loc[combined["frame_stratum"].eq("mixed")].empty:
    raise AssertionError("Source combined table unexpectedly has no mixed rows; exclusion check is not meaningful")

target_table["target_display"] = target_table["analysis_unit"]
target_table["frame_display"] = target_table["frame_stratum"].map(display_stratum)
target_table["b_se"] = target_table.apply(coefficient_plain, axis=1)
target_table["b_se_latex"] = target_table.apply(coefficient_latex, axis=1)
target_table["standardized_beta_display"] = target_table["standardized_beta_year"].map(lambda value: format_number(value, 2))
target_table["adj_r_squared_display"] = target_table["linear_adj_r_squared"].map(lambda value: format_number(value, 2))

table_export_columns = [
    "measure", "year_basis", "target_display", "frame_display", "term_role",
    "b_se", "standardized_beta_year", "standardized_beta_display",
    "linear_adj_r_squared", "adj_r_squared_display",
    "linear_slope_per_year", "linear_slope_se", "linear_p_value", "autocorrelation_flag",
    "ar1_sensitivity_slope_per_year", "ar1_sensitivity_p_value", "quadratic_delta_adj_r_squared",
]
target_table[table_export_columns].to_csv(TARGET_FRAME_CSV_PATH, index=False)

target_latex_rows = [
    [
        latex_escape(row.measure),
        latex_escape(row.target_display),
        latex_escape(row.frame_display),
        row.b_se_latex,
        row.standardized_beta_display,
        row.adj_r_squared_display,
    ]
    for row in target_table.itertuples(index=False)
]

target_note = (
    r"Note. Cells report annual unstandardised OLS slopes as $B(SE)$; $\beta$ is the standardised year coefficient. "
    r"Salience uses Common Crawl source year and has only an overall target row; semantic measures use document publication year. "
    r"Significance markers: $^{*}p<.05$, $^{**}p<.01$, $^{***}p<.001$. "
    r"$^{\dagger}$ indicates a residual-autocorrelation flag; AR(1) sensitivity estimates for flagged rows are reported in Appendix Table~\ref{tab:lsc-ar1-sensitivity-flagged}. "
    r"P values are descriptive and uncorrected."
)
target_latex = build_latex_table(
    caption="Descriptive annual trend models for ADHD and Autism LSC trajectories by frame.",
    label="tab:lsc-regression-target-frames",
    headers=["Measure", "Target", "Frame", r"$B(SE)$", r"$\beta$", r"Adj. $R^2$"],
    rows=target_latex_rows,
    alignment="lllccc",
    size="scriptsize",
    note=target_note,
)
TARGET_FRAME_TEX_PATH.write_text(target_latex)

print(f"Wrote {TARGET_FRAME_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TARGET_FRAME_TEX_PATH.relative_to(PROJECT_ROOT)}")
display(target_table[["measure", "target_display", "frame_display", "b_se", "standardized_beta_display", "adj_r_squared_display"]])


Wrote reports/tables/lsc/regression/lsc_regression_target_frames.csv
Wrote reports/tables/lsc/regression/lsc_regression_target_frames.tex


,measure,target_display,frame_display,b_se,standardized_beta_display,adj_r_squared_display
0,Salience,ADHD,Overall,0.0073 (0.0043),0.46,0.14
1,Salience,Autism,Overall,-0.0228* (0.0087),-0.62,0.33
2,Sentiment,ADHD,Overall,0.0012 (0.0007),0.48,0.16
3,Sentiment,ADHD,Clinical,0.0004^dagger (0.0009),0.15,-0.07
4,Sentiment,ADHD,Lived experience,0.0003 (0.0009),0.11,-0.08
5,Sentiment,Autism,Overall,0.0006 (0.0003),0.46,0.14
6,Sentiment,Autism,Clinical,0.0012^dagger (0.0009),0.36,0.05
7,Sentiment,Autism,Lived experience,-0.0019 (0.0012),-0.43,0.11
8,Intensity,ADHD,Overall,0.0010* (0.0004),0.63,0.35
9,Intensity,ADHD,Clinical,0.0013** (0.0004),0.69,0.43


## Appendix Baseline Table

The appendix table reports the three unframed comparator terms in the same style as the target-frame table. These rows keep the baseline contrast visible without overcrowding the main text table.


In [5]:
baseline_parts = []
for config in MEASURE_CONFIGS:
    measure_rows = combined.loc[combined["measure"].eq(config["measure"])]
    baseline_mask = measure_rows["term_role"].eq("baseline") & measure_rows["analysis_unit"].isin(BASELINE_UNITS)
    baseline_parts.append(measure_rows.loc[baseline_mask].copy())

baseline_table = pd.concat(baseline_parts, ignore_index=True)
baseline_table["unit_order"] = baseline_table["analysis_unit"].map({"frustration": 1, "loneliness": 2, "sadness": 3}).fillna(99)
baseline_table = baseline_table.sort_values(["measure_order", "unit_order"]).reset_index(drop=True)

if len(baseline_table) != 12:
    raise AssertionError(f"Baseline comparator regression table should have 12 rows, found {len(baseline_table)}")
if baseline_table["term_role"].ne("baseline").any():
    raise AssertionError("Appendix baseline table should contain baseline rows only")

baseline_table["comparator_display"] = baseline_table["analysis_unit"]
baseline_table["b_se"] = baseline_table.apply(coefficient_plain, axis=1)
baseline_table["b_se_latex"] = baseline_table.apply(coefficient_latex, axis=1)
baseline_table["standardized_beta_display"] = baseline_table["standardized_beta_year"].map(lambda value: format_number(value, 2))
baseline_table["adj_r_squared_display"] = baseline_table["linear_adj_r_squared"].map(lambda value: format_number(value, 2))

baseline_export_columns = [
    "measure", "year_basis", "comparator_display", "term_role",
    "b_se", "standardized_beta_year", "standardized_beta_display",
    "linear_adj_r_squared", "adj_r_squared_display",
    "linear_slope_per_year", "linear_slope_se", "linear_p_value", "autocorrelation_flag",
    "ar1_sensitivity_slope_per_year", "ar1_sensitivity_p_value", "quadratic_delta_adj_r_squared",
]
baseline_table[baseline_export_columns].to_csv(BASELINE_CSV_PATH, index=False)

baseline_latex_rows = [
    [
        latex_escape(row.measure),
        latex_escape(row.comparator_display),
        row.b_se_latex,
        row.standardized_beta_display,
        row.adj_r_squared_display,
    ]
    for row in baseline_table.itertuples(index=False)
]

baseline_note = (
    r"Note. Cells report annual unstandardised OLS slopes as $B(SE)$; $\beta$ is the standardised year coefficient. "
    r"Comparator terms are unframed baseline series. Salience uses Common Crawl source year and semantic measures use document publication year. "
    r"Significance markers: $^{*}p<.05$, $^{**}p<.01$, $^{***}p<.001$. "
    r"$^{\dagger}$ indicates a residual-autocorrelation flag; AR(1) sensitivity estimates for flagged rows are reported in Appendix Table~\ref{tab:lsc-ar1-sensitivity-flagged}. "
    r"P values are descriptive and uncorrected."
)
baseline_latex = build_latex_table(
    caption="Descriptive annual trend models for baseline comparator trajectories.",
    label="tab:lsc-regression-baseline-comparators",
    headers=["Measure", "Comparator", r"$B(SE)$", r"$\beta$", r"Adj. $R^2$"],
    rows=baseline_latex_rows,
    alignment="llccc",
    size="small",
    note=baseline_note,
)
BASELINE_TEX_PATH.write_text(baseline_latex)

print(f"Wrote {BASELINE_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {BASELINE_TEX_PATH.relative_to(PROJECT_ROOT)}")
display(baseline_table[["measure", "comparator_display", "b_se", "standardized_beta_display", "adj_r_squared_display"]])


Wrote reports/tables/lsc/regression/lsc_regression_baseline_comparators.csv
Wrote reports/tables/lsc/regression/lsc_regression_baseline_comparators.tex


,measure,comparator_display,b_se,standardized_beta_display,adj_r_squared_display
0,Salience,frustration,-0.0175^dagger (0.0167),-0.30,0.01
1,Salience,loneliness,0.0056* (0.0021),0.62,0.33
2,Salience,sadness,-0.0171** (0.0039),-0.80,0.61
3,Sentiment,frustration,0.0023*^dagger (0.0008),0.66,0.38
4,Sentiment,loneliness,0.0025*^dagger (0.0008),0.67,0.39
5,Sentiment,sadness,-0.0016** (0.0005),-0.69,0.42
6,Intensity,frustration,-0.0003^dagger (0.0002),-0.35,0.04
7,Intensity,loneliness,0.0001 (0.0004),0.06,-0.09
8,Intensity,sadness,0.0020*** (0.0003),0.92,0.83
9,Breadth,frustration,0.0020** (0.0006),0.72,0.48


## Appendix AR(1) Sensitivity Table

Rows marked with a dagger in the compact regression tables are collected into one self-contained appendix table. This avoids relying on supplementary CSV files for interpretation.


In [6]:
def format_ar1_slope(value: float, digits: int = 4) -> str:
    if pd.isna(value):
        return ""
    return f"${format_number(value, digits)}$"


def collect_ar1_rows() -> pd.DataFrame:
    rows = []

    for row in target_table.loc[target_table["autocorrelation_flag"]].itertuples(index=False):
        rows.append(
            {
                "source_table": r"Table~\ref{tab:lsc-regression-target-frames}",
                "measure": row.measure,
                "series": row.target_display,
                "frame": row.frame_display,
                "ols_b": row.linear_slope_per_year,
                "ar1_b": row.ar1_sensitivity_slope_per_year,
                "ar1_p": row.ar1_sensitivity_p_value,
                "digits": 4,
                "unit_note": "index_units_per_year",
                "sort_order": 100 + int(row.measure_order) * 10 + int(row.unit_order),
            }
        )

    for row in baseline_table.loc[baseline_table["autocorrelation_flag"]].itertuples(index=False):
        rows.append(
            {
                "source_table": r"Table~\ref{tab:lsc-regression-baseline-comparators}",
                "measure": row.measure,
                "series": row.comparator_display,
                "frame": "Baseline",
                "ols_b": row.linear_slope_per_year,
                "ar1_b": row.ar1_sensitivity_slope_per_year,
                "ar1_p": row.ar1_sensitivity_p_value,
                "digits": 4,
                "unit_note": "index_units_per_year",
                "sort_order": 300 + int(row.measure_order) * 10 + int(row.unit_order),
            }
        )

    if not CLASSIFICATION_TREND_CSV_PATH.exists():
        raise FileNotFoundError(f"Missing classification trend CSV: {CLASSIFICATION_TREND_CSV_PATH}")
    classification = pd.read_csv(CLASSIFICATION_TREND_CSV_PATH)
    for row in classification.loc[classification["autocorrelation_flag"].astype(bool)].itertuples(index=False):
        rows.append(
            {
                "source_table": r"Table~\ref{tab:lsc-classification-frame-time-trends}",
                "measure": "Frame classification",
                "series": row.analysis_unit,
                "frame": "Lived vs clinical",
                "ols_b": row.linear_slope_pp_per_year,
                "ar1_b": row.ar1_sensitivity_slope_pp_per_year,
                "ar1_p": row.ar1_sensitivity_p_value,
                "digits": 2,
                "unit_note": "percentage_points_per_year",
                "sort_order": 200,
            }
        )

    ar1 = pd.DataFrame(rows).sort_values("sort_order").reset_index(drop=True)
    if ar1.empty:
        raise AssertionError("Expected at least one residual-autocorrelation-flagged row for AR(1) reporting")
    return ar1


ar1_sensitivity = collect_ar1_rows()
ar1_sensitivity.to_csv(AR1_SENSITIVITY_CSV_PATH, index=False)

ar1_latex_rows = [
    [
        row.source_table,
        latex_escape(row.measure),
        latex_escape(row.series),
        latex_escape(row.frame),
        format_ar1_slope(row.ols_b, int(row.digits)),
        format_ar1_slope(row.ar1_b, int(row.digits)),
        format_p_value(row.ar1_p),
    ]
    for row in ar1_sensitivity.itertuples(index=False)
]

ar1_note = (
    r"Note. Rows are the series marked with $^{\dagger}$ in the compact regression tables. "
    r"OLS $B$ repeats the primary annual slope; AR(1) $B$ reports the first-order autoregressive sensitivity slope for the same series. "
    r"The frame-classification row is reported in percentage points per year; all other rows use the original index units per year. "
    r"P values are descriptive and uncorrected."
)
ar1_latex = build_latex_table(
    caption="AR(1) sensitivity models for autocorrelation-flagged annual series.",
    label="tab:lsc-ar1-sensitivity-flagged",
    headers=["Source", "Measure", "Series", "Frame", "OLS $B$", "AR(1) $B$", "AR(1) $p$"],
    rows=ar1_latex_rows,
    alignment="llllccc",
    size="scriptsize",
    note=ar1_note,
)
AR1_SENSITIVITY_TEX_PATH.write_text(ar1_latex)

print(f"Wrote {AR1_SENSITIVITY_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AR1_SENSITIVITY_TEX_PATH.relative_to(PROJECT_ROOT)}")
display(ar1_sensitivity[["source_table", "measure", "series", "frame", "ols_b", "ar1_b", "ar1_p"]])


Wrote reports/tables/lsc/diagnostics/lsc_ar1_sensitivity_flagged.csv
Wrote reports/tables/lsc/diagnostics/lsc_ar1_sensitivity_flagged.tex


,source_table,measure,series,frame,ols_b,ar1_b,ar1_p
0,Table~\ref{tab:lsc-regression-target-frames},Sentiment,ADHD,Clinical,0.000427,0.001464,0.294530
1,Table~\ref{tab:lsc-regression-target-frames},Sentiment,Autism,Clinical,0.001154,0.002507,0.077325
2,Table~\ref{tab:lsc-regression-target-frames},Breadth,ADHD,Lived experience,-0.000366,-0.000437,0.246934
3,Table~\ref{tab:lsc-regression-target-frames},Breadth,Autism,Overall,0.000119,-0.000133,0.752659
4,Table~\ref{tab:lsc-regression-target-frames},Breadth,Autism,Clinical,0.000749,0.000157,0.754792
5,Table~\ref{tab:lsc-classification-frame-time-t...,Frame classification,Autism,Lived vs clinical,0.585181,0.229514,0.370448
6,Table~\ref{tab:lsc-regression-baseline-compara...,Salience,frustration,Baseline,-0.017517,0.015957,0.538315
7,Table~\ref{tab:lsc-regression-baseline-compara...,Sentiment,frustration,Baseline,0.002287,0.007158,0.002728
8,Table~\ref{tab:lsc-regression-baseline-compara...,Sentiment,loneliness,Baseline,0.002453,0.003889,0.009703
9,Table~\ref{tab:lsc-regression-baseline-compara...,Intensity,frustration,Baseline,-0.000261,-0.000999,0.049705


## Handoff

The CSV exports preserve machine-readable values for later inspection. The LaTeX exports are intended as `booktabs` table fragments: the target-frame table belongs in the Results chapter and the baseline comparator table belongs in the appendix.


In [7]:
print("Saved regression table outputs:")
for path in [COMBINED_PATH, TARGET_FRAME_CSV_PATH, TARGET_FRAME_TEX_PATH, BASELINE_CSV_PATH, BASELINE_TEX_PATH, AR1_SENSITIVITY_CSV_PATH, AR1_SENSITIVITY_TEX_PATH]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

print("\nTarget-frame table rows:", len(target_table))
print("Baseline comparator table rows:", len(baseline_table))
print("Mixed rows in target-frame table:", int(target_table["frame_stratum"].eq("mixed").sum()))
print("Baseline rows in target-frame table:", int(target_table["term_role"].eq("baseline").sum()))
print("AR(1) sensitivity table rows:", len(ar1_sensitivity))


Saved regression table outputs:
- reports/tables/lsc/regression/lsc_regression_models_combined.csv
- reports/tables/lsc/regression/lsc_regression_target_frames.csv
- reports/tables/lsc/regression/lsc_regression_target_frames.tex
- reports/tables/lsc/regression/lsc_regression_baseline_comparators.csv
- reports/tables/lsc/regression/lsc_regression_baseline_comparators.tex
- reports/tables/lsc/diagnostics/lsc_ar1_sensitivity_flagged.csv
- reports/tables/lsc/diagnostics/lsc_ar1_sensitivity_flagged.tex

Target-frame table rows: 20
Baseline comparator table rows: 12
Mixed rows in target-frame table: 0
Baseline rows in target-frame table: 0
AR(1) sensitivity table rows: 11
